In [1]:
# ── TRAIN LSTM ON WINDOWED REALISTIC DATA ─────────────────────────────────
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix
import matplotlib.pyplot as plt
import time

# Set device
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print(" Using MPS")
else:
    device = torch.device("cpu")
    print(" Using CPU")

# Load windowed data
DATA_DIR = Path("data/windowed_realistic")
data = np.load(DATA_DIR / "windowed_realistic_splits.npz")

X_train = data['X_train']  # (11330, 200, 129)
X_val = data['X_val']      # (9189, 200, 129)
X_test = data['X_test']    # (9133, 200, 129)
y_val = data['y_val']
y_test = data['y_test']

print(f"Data shapes:")
print(f"  Train: {X_train.shape}")
print(f"  Val:   {X_val.shape}")
print(f"  Test:  {X_test.shape}")
print(f"  Test anomaly rate: {y_test.mean():.2%}")

 Using MPS
Data shapes:
  Train: (11330, 200, 129)
  Val:   (9189, 200, 129)
  Test:  (9133, 200, 129)
  Test anomaly rate: 1.02%


In [2]:
# ── SIMPLE LSTM MODEL FOR WINDOWED DATA ───────────────────────────────────
class WindowedLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, latent_dim=16):
        super().__init__()
        self.encoder = nn.LSTM(input_dim, hidden_dim, 1, batch_first=True)
        self.fc_e = nn.Linear(hidden_dim, latent_dim)
        self.fc_d = nn.Linear(latent_dim, hidden_dim)
        self.decoder = nn.LSTM(hidden_dim, hidden_dim, 1, batch_first=True)
        self.output = nn.Linear(hidden_dim, input_dim)
    
    def forward(self, x):
        enc_out, _ = self.encoder(x)
        latent = self.fc_e(enc_out[:, -1, :])
        latent_exp = latent.unsqueeze(1).repeat(1, x.shape[1], 1)
        dec_in = self.fc_d(latent_exp)
        dec_out, _ = self.decoder(dec_in)
        recon = self.output(dec_out)
        return recon, latent

# Initialize
input_dim = X_train.shape[2]
model = WindowedLSTM(input_dim=input_dim).to(device)

print(f"Model: LSTM Autoencoder")
print(f"  Input: {input_dim} features")
print(f"  Window: {X_train.shape[1]} timesteps")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

Model: LSTM Autoencoder
  Input: 129 features
  Window: 200 timesteps
  Parameters: 93,713


In [3]:
# ── TRAINING ──────────────────────────────────────────────────────────────
# Convert to tensors
X_train_tensor = torch.FloatTensor(X_train).to(device)
X_val_tensor = torch.FloatTensor(X_val).to(device)

# DataLoader
batch_size = 64  # Can use larger batch size with windowed data
train_loader = DataLoader(TensorDataset(X_train_tensor), batch_size=batch_size, shuffle=True)

# Training setup
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

print(f"Training with {len(train_loader)} batches of size {batch_size}")

Training with 178 batches of size 64


In [4]:
# ── TRAINING LOOP ─────────────────────────────────────────────────────────
num_epochs = 30
best_val_auc = 0
patience = 5
patience_counter = 0

train_losses = []
val_losses = []
val_aucs = []

print("\nStarting training...")
start_time = time.time()

for epoch in range(num_epochs):
    # Training
    model.train()
    train_loss = 0
    for batch in train_loader:
        X_batch = batch[0]
        optimizer.zero_grad()
        recon, _ = model(X_batch)
        loss = criterion(recon, X_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    avg_train_loss = train_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    
    # Validation
    model.eval()
    val_loss = 0
    val_errors = []
    
    with torch.no_grad():
        for i in range(0, len(X_val_tensor), batch_size):
            batch = X_val_tensor[i:i+batch_size]
            recon, _ = model(batch)
            loss = criterion(recon, batch)
            val_loss += loss.item() * len(batch)
            
            mse = torch.mean((recon - batch) ** 2, dim=(1, 2))
            val_errors.extend(mse.cpu().numpy())
    
    avg_val_loss = val_loss / len(X_val_tensor)
    val_losses.append(avg_val_loss)
    
    # Calculate AUC
    val_auc = roc_auc_score(y_val, val_errors)
    val_aucs.append(val_auc)
    
    # Learning rate scheduling
    scheduler.step(avg_val_loss)
    
    # Save best model
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        patience_counter = 0
        torch.save(model.state_dict(), DATA_DIR / "best_windowed_model.pth")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break
    
    if (epoch + 1) % 5 == 0:
        elapsed = time.time() - start_time
        print(f"Epoch {epoch+1}: Train Loss={avg_train_loss:.4f}, Val Loss={avg_val_loss:.4f}, Val AUC={val_auc:.4f}, Time={elapsed:.1f}s")

print(f"\n Training complete! Best Val AUC: {best_val_auc:.4f}")


Starting training...
Epoch 5: Train Loss=0.2951, Val Loss=0.4877, Val AUC=0.5187, Time=28.5s
Epoch 10: Train Loss=0.2265, Val Loss=0.2454, Val AUC=0.5669, Time=52.5s
Early stopping at epoch 13

 Training complete! Best Val AUC: 0.5692


In [5]:
# ── EVALUATE ON TEST SET ──────────────────────────────────────────────────
# Load best model
model.load_state_dict(torch.load(DATA_DIR / "best_windowed_model.pth"))
model.eval()

# Get test errors
test_errors = []
with torch.no_grad():
    X_test_tensor = torch.FloatTensor(X_test).to(device)
    for i in range(0, len(X_test_tensor), batch_size):
        batch = X_test_tensor[i:i+batch_size]
        recon, _ = model(batch)
        mse = torch.mean((recon - batch) ** 2, dim=(1, 2))
        test_errors.extend(mse.cpu().numpy())

test_errors = np.array(test_errors)

# Metrics
test_auc = roc_auc_score(y_test, test_errors)
test_pr_auc = average_precision_score(y_test, test_errors)

print("\n" + "="*60)
print("TEST SET PERFORMANCE (Realistic 1% Anomalies)")
print("="*60)
print(f"ROC-AUC: {test_auc:.4f}")
print(f"PR-AUC:  {test_pr_auc:.4f}")

# Find optimal threshold from validation
from sklearn.metrics import roc_curve
val_errors = []
with torch.no_grad():
    for i in range(0, len(X_val_tensor), batch_size):
        batch = X_val_tensor[i:i+batch_size]
        recon, _ = model(batch)
        mse = torch.mean((recon - batch) ** 2, dim=(1, 2))
        val_errors.extend(mse.cpu().numpy())

fpr, tpr, thresholds = roc_curve(y_val, val_errors)
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]

# Apply threshold
test_pred = (test_errors > optimal_threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, test_pred).ravel()

print(f"\nOptimal threshold: {optimal_threshold:.4f}")
print(f"\nConfusion Matrix (on {len(y_test)} test windows):")
print(f"  TP (caught anomalies): {tp}")
print(f"  FN (missed anomalies): {fn}")
print(f"  FP (false alarms): {fp}")
print(f"  TN (correct normals): {tn}")

print(f"\nMetrics:")
print(f"  Anomaly Recall: {tp/(tp+fn):.4f} ({tp}/{tp+fn})")
print(f"  Anomaly Precision: {tp/(tp+fp):.4f}")
print(f"  F1-Score: {2*tp/(2*tp+fp+fn):.4f}")


TEST SET PERFORMANCE (Realistic 1% Anomalies)
ROC-AUC: 0.5448
PR-AUC:  0.0303

Optimal threshold: 0.2917

Confusion Matrix (on 9133 test windows):
  TP (caught anomalies): 34
  FN (missed anomalies): 59
  FP (false alarms): 2368
  TN (correct normals): 6672

Metrics:
  Anomaly Recall: 0.3656 (34/93)
  Anomaly Precision: 0.0142
  F1-Score: 0.0273
